# 01 – Data Understanding (EDA)
Aucune donnée n'est modifiée. Tous les chiffres sont calculés par le code.

In [ ]:
import sys, json
sys.path.append('..')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.load_data import load_raw_data, file_sha256, TARGET
from src.data.validate_data import (quality_report, class_distribution,
    descriptive_stats, iqr_outlier_summary, correlation_with_target)

ROOT = Path('..').resolve()
FIG = ROOT / 'reports' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
MET = ROOT / 'reports' / 'metrics'; MET.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
SUMMARY = {}
def save(name):
    plt.tight_layout(); plt.savefig(FIG / name, dpi=130); plt.show()


## 1. Structure du dataset

In [ ]:
RAW = ROOT / 'data' / 'raw' / 'creditcard.csv'
df = load_raw_data(RAW)
SUMMARY['sha256'] = file_sha256(RAW)
print('SHA-256 :', SUMMARY['sha256'])
print('Dimensions :', df.shape)
print('Cible :', TARGET)
display(df.head())
display(df.dtypes.to_frame('dtype').T)


## 2. Qualité des données

In [ ]:
qr = quality_report(df, TARGET)
SUMMARY['quality'] = qr
for k, v in qr.items(): print(f'{k:28s}: {v}')


In [ ]:
# Doublons : sont-ils des transactions identiques avec labels différents ?
dups = df[df.duplicated(keep=False)]
print('Lignes impliquées dans des doublons exacts :', len(dups))
print('Répartition des classes parmi les doublons exacts :')
print(df[df.duplicated()][TARGET].value_counts())
conflict = df.drop(columns=[TARGET]).duplicated(keep=False) & ~df.duplicated(keep=False)
print('Doublons de features avec label différent :', int(conflict.sum()))
SUMMARY['duplicates_by_class'] = df[df.duplicated()][TARGET].value_counts().to_dict()


## 3. Distribution de la fraude

In [ ]:
cd = class_distribution(df, TARGET)
SUMMARY['class_distribution'] = cd
print(cd)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x=TARGET, data=df, ax=ax[0]); ax[0].set_title('Distribution de la cible')
sns.countplot(x=TARGET, data=df, ax=ax[1]); ax[1].set_yscale('log'); ax[1].set_title('Échelle logarithmique')
save('01_target_distribution.png')


## 4. Analyse statistique

In [ ]:
stats = descriptive_stats(df, TARGET)
display(stats[['mean','median','std','min','1%','25%','75%','99%','max','skewness','kurtosis']].round(3))
SUMMARY['top_skew'] = stats['skewness'].abs().sort_values(ascending=False).head(8).round(3).to_dict()
SUMMARY['top_kurtosis'] = stats['kurtosis'].sort_values(ascending=False).head(8).round(3).to_dict()
print('Top asymétrie :', SUMMARY['top_skew'])
print('Top kurtosis  :', SUMMARY['top_kurtosis'])
for cls in (0, 1):
    SUMMARY[f'amount_class_{cls}'] = df.loc[df[TARGET]==cls, 'Amount'].describe().round(3).to_dict()
    print('Amount, classe', cls, SUMMARY[f'amount_class_{cls}'])


## 5. Distributions des variables numériques

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df['Amount'], bins=80, ax=ax[0]); ax[0].set_title('Amount')
sns.histplot(np.log1p(df['Amount']), bins=80, ax=ax[1]); ax[1].set_title('log1p(Amount)')
sns.histplot(df['Time'], bins=80, ax=ax[2]); ax[2].set_title('Time (secondes)')
save('02_amount_time_distributions.png')


In [ ]:
vcols = [c for c in df.columns if c.startswith('V')]
fig, axes = plt.subplots(7, 4, figsize=(16, 20))
for c, a in zip(vcols, axes.ravel()):
    sns.histplot(df[c], bins=60, ax=a, color='steelblue'); a.set_title(c); a.set_xlabel('')
save('03_v_features_histograms.png')


## 6. Boxplots et outliers (IQR)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(x=TARGET, y='Amount', data=df, ax=ax[0]); ax[0].set_title('Amount par classe')
sns.boxplot(x=TARGET, y=np.log1p(df['Amount']), data=df, ax=ax[1]); ax[1].set_title('log1p(Amount) par classe')
save('04_amount_boxplots.png')


In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(16, 22))
for c, a in zip(vcols, axes.ravel()):
    sns.boxplot(x=TARGET, y=c, data=df, ax=a, fliersize=1); a.set_title(c)
save('05_v_features_boxplots_by_class.png')


In [ ]:
out = iqr_outlier_summary(df, TARGET)
display(out.sort_values('outlier_pct_all', ascending=False).head(10).round(2))
SUMMARY['outliers_top'] = out.sort_values('outlier_pct_all', ascending=False).head(8).round(2).to_dict('index')
o = out.sort_values('outlier_pct_fraud', ascending=False)
fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(o)); w = 0.4
ax.bar(x - w/2, o['outlier_pct_legit'], w, label='Légitime')
ax.bar(x + w/2, o['outlier_pct_fraud'], w, label='Fraude')
ax.set_xticks(x); ax.set_xticklabels(o.index, rotation=90)
ax.set_ylabel('% hors IQR'); ax.set_title('Outliers IQR par variable et par classe'); ax.legend()
save('06_outliers_by_class.png')


## 7. Fraude vs légitime

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=df, x=np.log1p(df['Amount']), hue=TARGET, common_norm=False, ax=ax[0]); ax[0].set_title('log1p(Amount) par classe')
df['_hour'] = (df['Time'] // 3600) % 24
rate = df.groupby('_hour')[TARGET].mean() * 100
rate.plot(kind='bar', ax=ax[1]); ax[1].set_title('Taux de fraude (%) par heure (Time mod 24h)'); ax[1].set_xlabel('heure')
save('07_fraud_vs_legit.png')
SUMMARY['time_span_hours'] = round(float(df['Time'].max() - df['Time'].min()) / 3600, 2)
SUMMARY['fraud_rate_by_hour_max'] = {'hour': int(rate.idxmax()), 'pct': round(float(rate.max()), 4)}
print('Durée couverte (h) :', SUMMARY['time_span_hours'])
df.drop(columns='_hour', inplace=True)


In [ ]:
vsel = correlation_with_target(df, TARGET).head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for c, a in zip(vsel, axes.ravel()):
    sns.kdeplot(data=df, x=c, hue=TARGET, common_norm=False, ax=a); a.set_title(c)
save('08_top_features_by_class.png')


## 8. Corrélations et fuite de données potentielle

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax); ax.set_title('Matrice de corrélation')
save('09_correlation_matrix.png')
ct = correlation_with_target(df, TARGET)
SUMMARY['top_target_corr'] = ct.head(10).round(4).to_dict()
print(ct.head(10).round(4))
off = corr.where(~np.eye(len(corr), dtype=bool)).drop(index=TARGET, columns=TARGET).abs().stack()
SUMMARY['max_feature_feature_corr'] = {'pair': list(off.idxmax()), 'value': round(float(off.max()), 4)}
print('Plus forte corrélation feature-feature :', SUMMARY['max_feature_feature_corr'])
print('Features avec |corr| > 0.9 avec la cible :', [c for c, v in ct.items() if abs(v) > 0.9])


## 9. Sauvegarde du résumé (source des chiffres du rapport)

In [ ]:
with open(MET / 'eda_summary.json', 'w') as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print('Sauvé :', MET / 'eda_summary.json')
